In [2]:
import os
import cv2
import numpy as np
import shutil
from sklearn.model_selection import train_test_split

os.chdir('/Users/badisasaisriharsha/Desktop/Glaucoma-Detection-ResCapsNet')
print("Working dir:", os.getcwd())

Working dir: /Users/badisasaisriharsha/Desktop/Glaucoma-Detection-ResCapsNet


In [3]:
def apply_clahe(img_path, output_path, clip_limit=2.0, tile_grid=(8, 8), size=(256, 256)):
    img = cv2.imread(img_path)
    if img is None:
        return False
    img = cv2.resize(img, size)
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid)
    l_clahe = clahe.apply(l)
    lab_clahe = cv2.merge([l_clahe, a, b])
    result = cv2.cvtColor(lab_clahe, cv2.COLOR_LAB2BGR)
    cv2.imwrite(output_path, result)
    return True

def clahe_copy(src_list, dst_dir, prefix):
    skipped = 0
    for i, src in enumerate(src_list):
        dst = os.path.join(dst_dir, f"{prefix}_{i:04d}.jpg")
        if not apply_clahe(src, dst):
            skipped += 1
    if skipped:
        print(f"  ⚠️  {prefix}: skipped {skipped} unreadable files")

print("CLAHE function ready")

CLAHE function ready


In [4]:
out_train_pos = 'data/processed/combined/Train/Glaucoma_Positive'
out_train_neg = 'data/processed/combined/Train/Glaucoma_Negative'
out_val_pos   = 'data/processed/combined/Val/Glaucoma_Positive'
out_val_neg   = 'data/processed/combined/Val/Glaucoma_Negative'

for d in [out_train_pos, out_train_neg, out_val_pos, out_val_neg]:
    os.makedirs(d, exist_ok=True)

print("Output dirs ready")

Output dirs ready


In [6]:
# Cell 4 — Count Kaggle source
kaggle_train_pos = 'data/processed/Train/Glaucoma_Positive'
kaggle_train_neg = 'data/processed/Train/Glaucoma_Negative'
kaggle_val_pos   = 'data/processed/Validation/Glaucoma_Positive'
kaggle_val_neg   = 'data/processed/Validation/Glaucoma_Negative'

def count_dir(d):
    return len([f for f in os.listdir(d) if f.lower().endswith(('.jpg','.jpeg','.png'))])

print(f"Kaggle Train POS: {count_dir(kaggle_train_pos)}, NEG: {count_dir(kaggle_train_neg)}")
print(f"Kaggle Val   POS: {count_dir(kaggle_val_pos)},  NEG: {count_dir(kaggle_val_neg)}")

Kaggle Train POS: 134, NEG: 386
Kaggle Val   POS: 34,  NEG: 96


In [7]:
# Cell 5 — Count ACRIMA source
acrima_root = 'data/raw/ACRIMA'
acrima_pos, acrima_neg = [], []

for split in ['train', 'test']:
    for fname in os.listdir(os.path.join(acrima_root, split, 'Glaucoma')):
        if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            acrima_pos.append(os.path.join(acrima_root, split, 'Glaucoma', fname))
    for fname in os.listdir(os.path.join(acrima_root, split, 'Non Glaucoma')):
        if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            acrima_neg.append(os.path.join(acrima_root, split, 'Non Glaucoma', fname))

print(f"ACRIMA total POS: {len(acrima_pos)}, NEG: {len(acrima_neg)}")

ACRIMA total POS: 396, NEG: 309


In [8]:
# Cell 6 — Count + collect DRISHTI source
drishti_train_root = 'data/raw/DRISHTI/Training-20211018T055246Z-001/Training/Images'
drishti_test_root  = 'data/raw/DRISHTI/Test-20211018T060000Z-001/Test/Images'

drishti_pos, drishti_neg = [], []

for f in os.listdir(os.path.join(drishti_train_root, 'GLAUCOMA')):
    if f.lower().endswith(('.jpg','.jpeg','.png')):
        drishti_pos.append(os.path.join(drishti_train_root, 'GLAUCOMA', f))

for f in os.listdir(os.path.join(drishti_train_root, 'NORMAL')):
    if f.lower().endswith(('.jpg','.jpeg','.png')):
        drishti_neg.append(os.path.join(drishti_train_root, 'NORMAL', f))

for f in os.listdir(os.path.join(drishti_test_root, 'glaucoma')):
    if f.lower().endswith(('.jpg','.jpeg','.png')):
        drishti_pos.append(os.path.join(drishti_test_root, 'glaucoma', f))

for f in os.listdir(os.path.join(drishti_test_root, 'normal')):
    if f.lower().endswith(('.jpg','.jpeg','.png')):
        drishti_neg.append(os.path.join(drishti_test_root, 'normal', f))

print(f"DRISHTI pooled POS: {len(drishti_pos)}, NEG: {len(drishti_neg)}")

DRISHTI pooled POS: 70, NEG: 31


In [9]:
# Cell 7 — CLAHE + copy all three sources
# DRISHTI split + CLAHE
d_pos_train, d_pos_val = train_test_split(drishti_pos, test_size=0.2, random_state=42)
d_neg_train, d_neg_val = train_test_split(drishti_neg, test_size=0.2, random_state=42)

clahe_copy(d_pos_train, out_train_pos, 'drishti_pos')
clahe_copy(d_neg_train, out_train_neg, 'drishti_neg')
clahe_copy(d_pos_val,   out_val_pos,   'drishti_pos_val')
clahe_copy(d_neg_val,   out_val_neg,   'drishti_neg_val')
print("DRISHTI done")

# ACRIMA split + CLAHE
a_pos_train, a_pos_val = train_test_split(acrima_pos, test_size=0.2, random_state=42)
a_neg_train, a_neg_val = train_test_split(acrima_neg, test_size=0.2, random_state=42)

clahe_copy(a_pos_train, out_train_pos, 'acrima_pos')
clahe_copy(a_neg_train, out_train_neg, 'acrima_neg')
clahe_copy(a_pos_val,   out_val_pos,   'acrima_pos_val')
clahe_copy(a_neg_val,   out_val_neg,   'acrima_neg_val')
print("ACRIMA done")

# Kaggle — already CLAHE'd, just copy
def copy_existing(src_dir, dst_dir, prefix):
    files = [f for f in os.listdir(src_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))]
    for i, f in enumerate(files):
        shutil.copy2(os.path.join(src_dir, f), os.path.join(dst_dir, f"{prefix}_{i:04d}.jpg"))

copy_existing(kaggle_train_pos, out_train_pos, 'kaggle_pos')
copy_existing(kaggle_train_neg, out_train_neg, 'kaggle_neg')
copy_existing(kaggle_val_pos,   out_val_pos,   'kaggle_pos_val')
copy_existing(kaggle_val_neg,   out_val_neg,   'kaggle_neg_val')
print("Kaggle done")

DRISHTI done
ACRIMA done
Kaggle done


In [10]:
# Cell 8 — Final counts + focal alpha
train_pos = count_dir(out_train_pos)
train_neg = count_dir(out_train_neg)
val_pos   = count_dir(out_val_pos)
val_neg   = count_dir(out_val_neg)
total_train = train_pos + train_neg

alpha = round(train_neg / total_train, 4)

print(f"Train POS: {train_pos}  NEG: {train_neg}  Total: {total_train}  Ratio: 1:{train_neg/train_pos:.2f}")
print(f"Val   POS: {val_pos}   NEG: {val_neg}")
print(f"Focal α: {alpha}  ← record in results.md")

Train POS: 506  NEG: 657  Total: 1163  Ratio: 1:1.30
Val   POS: 128   NEG: 165
Focal α: 0.5649  ← record in results.md


In [12]:
# Cell 9 — Dataset summary 
print("=" * 50)
print("FINAL DATASET COUNTS")
print("=" * 50)
print(f"Train POS: 506  NEG: 657  Total: 1163  Ratio: 1:1.30")
print(f"Val   POS: 128  NEG: 165  Total: 293")
print()
print("Sources:")
print("  Kaggle  — Train POS 134 / NEG 386 | Val POS 34 / NEG 96")
print("  DRISHTI — pooled 70 POS / 31 NEG  | split 80/20")
print("  ACRIMA  — pooled 396 POS / 309 NEG | split 80/20")
print()
print("Focal Loss α = 0.5649  (NEG/Total train)")
print("Ratio improved from 1:2.9 (Kaggle-only) → 1:1.30 (combined)")
print("=" * 50)

FINAL DATASET COUNTS
Train POS: 506  NEG: 657  Total: 1163  Ratio: 1:1.30
Val   POS: 128  NEG: 165  Total: 293

Sources:
  Kaggle  — Train POS 134 / NEG 386 | Val POS 34 / NEG 96
  DRISHTI — pooled 70 POS / 31 NEG  | split 80/20
  ACRIMA  — pooled 396 POS / 309 NEG | split 80/20

Focal Loss α = 0.5649  (NEG/Total train)
Ratio improved from 1:2.9 (Kaggle-only) → 1:1.30 (combined)
